# Notebook 03: Preprocessing and training

> Switch to a GPU runtime before running (Runtime > Change runtime type > T4 GPU).

Full fine-tune of ImageNet-pretrained backbones at a low learning rate. Uses an identical protocol in both transfer directions so any observed asymmetry is not a procedural artifact.

Executes seed 42 first (4 runs) to validate the pipeline, then uses RUN_FULL to scale to the full study design:
- K2N: 2 architectures x 3 sizes x 3 seeds = 18 fits
- N2K: 2 architectures x 5 folds x 3 seeds = 30 fits

## 1. Setup and GPU Check

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

import sys
from pathlib import Path

REPO_ON_DRIVE = "/content/drive/MyDrive/MSc AI DISSERTATION/crosspop-cxr-asymmetry"
for candidate in [REPO_ON_DRIVE, "..", "."]:
    p = Path(candidate).resolve()
    if (p / "config.py").exists():
        sys.path.insert(0, str(p))
        break

import importlib
import json
import time
import pandas as pd
import torch

import config
from src import splits as SP
from src import datasets as DS
from src import models as M
from src import train as TR

for mod in (config, SP, DS, M, TR):
    importlib.reload(mod)

config.ensure_output_dirs()
config.summary()


Mounted at /content/drive
Cross-Population CXR Asymmetry Experiment Configuration
-------------------------------------------------------
DATA_ROOT           : /content/drive/MyDrive/MSc AI DISSERTATION/data
Architectures       : ['mobilenet_v2', 'efficientnet_b0']
Directions          : ['K2N', 'N2K']
Matched sizes       : [50, 100, 190]
Class balance       : 1:1 Ratio
Total training runs : 12 (2 archs x 2 dirs x 3 sizes)
CV folds            : 5 | Holdout: 30
Seeds               : [42, 43, 44, 45, 46, 47]


In [ ]:
device = config.get_device()
if device == "cpu":
    print("\n>>> WARNING: Running on CPU. Switch runtime to T4 GPU for faster training.")
else:
    print(f"\nGPU active: {torch.cuda.get_device_name(0)}")


Compute Device: CPU

>>> WARNING: Running on CPU. Switch runtime to T4 GPU for faster training.


In [ ]:
manifest_path = config.RESULTS_DIR / "manifests" / "splits_all.csv"
manifest = pd.read_csv(manifest_path)
print(f"Loaded manifest: {manifest.shape} from {manifest_path.name}")
print(manifest.groupby(["dataset","direction","split"]).size().to_string())

Loaded manifest: (7644, 10) from splits_all.csv
dataset  direction  split  
kermany  K2N        train      2040
         N2K        test        624
togunwa  K2N        holdout     180
         N2K        train      3840
                    val         960


## 2. Dataset Builders per Direction

In [ ]:
def make_datasets_K2N(manifest, size, seed, val_frac=0.2):
    """Build train and validation datasets for Kermany-to-Togunwa (K2N)."""
    rows = manifest[
        (manifest.dataset == "kermany") &
        (manifest.split == "train") &
        (manifest["size"] == size) &
        (manifest.seed.astype(str) == str(seed))
    ]
    recs = rows[["path", "label", "class"]].to_dict("records")

    # SP.stratified_holdout returns (holdout, working) -> (val_recs, train_recs)
    val_recs, train_recs = SP.stratified_holdout(recs, round(len(recs) * val_frac), seed)

    tf_train = DS.build_transforms(config.IMAGE_SIZE, config.IMAGENET_MEAN, config.IMAGENET_STD, train=True)
    tf_eval = DS.build_transforms(config.IMAGE_SIZE, config.IMAGENET_MEAN, config.IMAGENET_STD, train=False)

    train_ds = DS.CXRDataset(pd.DataFrame(train_recs), tf_train)
    val_ds = DS.CXRDataset(pd.DataFrame(val_recs), tf_eval)
    return train_ds, val_ds, len(train_recs), len(val_recs)

def make_datasets_N2K(manifest, fold, seed):
    """Build train and validation datasets for Togunwa-to-Kermany (N2K) cross-validation."""
    base = manifest[
        (manifest.dataset == "togunwa") &
        (manifest.direction == "N2K") &
        (manifest.seed.astype(str) == str(seed)) &
        (manifest.fold == fold)
    ]
    tr_recs = base[base.split == "train"][["path", "label", "class"]].to_dict("records")
    va_recs = base[base.split == "val"][["path", "label", "class"]].to_dict("records")

    tf_train = DS.build_transforms(config.IMAGE_SIZE, config.IMAGENET_MEAN, config.IMAGENET_STD, train=True)
    tf_eval = DS.build_transforms(config.IMAGE_SIZE, config.IMAGENET_MEAN, config.IMAGENET_STD, train=False)

    train_ds = DS.CXRDataset(pd.DataFrame(tr_recs), tf_train)
    val_ds = DS.CXRDataset(pd.DataFrame(va_recs), tf_eval)
    return train_ds, val_ds, len(tr_recs), len(va_recs)


## 3. Model Training Execution Routine

In [ ]:
def run_one(arch, direction, size, seed, fold=None, batch_size=None, verbose=True, force=False):
    """Execute training for a single architecture and experimental configuration.

    If a checkpoint (.pth) and its metadata (.json) already exist for this exact
    tag, training is skipped and the existing metadata record is returned instead
    -- unless force=True. This lets the full config loop in section 5 be re-run
    safely after extending config.SEEDS: previously-trained seeds are skipped,
    only newly-added seeds actually train.
    """
    if direction == "K2N":
        tag = f"{arch}__K2N__size{size}__seed{seed}"
    else:
        fold = 0 if fold is None else fold
        tag = f"{arch}__N2K__size190__seed{seed}__fold{fold}"

    ckpt = config.CHECKPOINTS_DIR / f"{tag}.pth"
    json_path = config.CHECKPOINTS_DIR / f"{tag}.json"

    if ckpt.exists() and json_path.exists() and not force:
        if verbose:
            print(f"[skip] {tag} already trained, loading existing metadata")
        with open(json_path) as f:
            return json.load(f)

    config.set_all_seeds(seed)

    if direction == "K2N":
        train_ds, val_ds, n_tr, n_va = make_datasets_K2N(manifest, size, seed)
    else:
        train_ds, val_ds, n_tr, n_va = make_datasets_N2K(manifest, fold, seed)

    bs = batch_size or min(config.BATCH_SIZE, max(4, n_tr // 4))
    if verbose:
        print(f"\n--- Running [{tag}] ---")
        print(f"Train samples: {n_tr} | Val samples: {n_va} | Batch size: {bs}")

    model = M.build_model(arch, pretrained=True)
    t0 = time.time()

    model, info = TR.train_one(
        model, train_ds, val_ds, device,
        lr=config.LEARNING_RATE,
        weight_decay=config.WEIGHT_DECAY,
        max_epochs=config.MAX_EPOCHS,
        patience=config.EARLY_STOPPING_PATIENCE,
        batch_size=bs,
        mixed_precision=config.MIXED_PRECISION,
        verbose=verbose
    )
    secs = time.time() - t0

    torch.save(model.state_dict(), ckpt)

    rec = {
        "tag": tag,
        "arch": arch,
        "direction": direction,
        "size": size,
        "seed": seed,
        "fold": fold,
        "n_train": n_tr,
        "n_val": n_va,
        "batch_size": bs,
        "best_epoch": info["best_epoch"],
        "best_val_loss": info["best_val_loss"],
        "seconds": round(secs, 1),
        "checkpoint": str(ckpt)
    }

    with open(json_path, "w") as f:
        json.dump(rec, f, indent=2)

    if verbose:
        print(f"Saved checkpoint: {ckpt.name} ({secs:.1f}s)")

    return rec


## 4. Pipeline Validation (Seed 42 Runs)

In [ ]:
print("\n=========================================")
print("Executing Seed 42 Pipeline Validation")
print("=========================================")

validation_records = []
for arch in config.ARCHITECTURES:
    validation_records.append(run_one(arch, "K2N", size=190, seed=42))
    validation_records.append(run_one(arch, "N2K", size=190, seed=42, fold=0))

val_df = pd.DataFrame(validation_records)[["tag", "n_train", "n_val", "best_epoch", "best_val_loss", "seconds"]]
print("\nValidation Runs Summary:")
print(val_df.to_string(index=False))



Executing Seed 42 Pipeline Validation

--- Running [mobilenet_v2__K2N__size190__seed42] ---
Train samples: 152 | Val samples: 38 | Batch size: 16
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 105MB/s] 


    Epoch  0 | Train Loss: 0.6053 | Val Loss: 0.4101  *
    Epoch  1 | Train Loss: 0.2568 | Val Loss: 0.2121  *
    Epoch  2 | Train Loss: 0.1186 | Val Loss: 0.2329
    Epoch  3 | Train Loss: 0.0638 | Val Loss: 0.1282  *
    Epoch  4 | Train Loss: 0.1442 | Val Loss: 0.0865  *
    Epoch  5 | Train Loss: 0.0523 | Val Loss: 0.1265
    Epoch  6 | Train Loss: 0.0358 | Val Loss: 0.1437
    Epoch  7 | Train Loss: 0.1233 | Val Loss: 0.1069
    Epoch  8 | Train Loss: 0.0287 | Val Loss: 0.1313
    Epoch  9 | Train Loss: 0.0339 | Val Loss: 0.1102
    Early stopping triggered at epoch 9 (Best epoch: 4, Val Loss: 0.0865)
Saved checkpoint: mobilenet_v2__K2N__size190__seed42.pth (146.2s)

--- Running [mobilenet_v2__N2K__size190__seed42__fold0] ---
Train samples: 127 | Val samples: 33 | Batch size: 16
    Epoch  0 | Train Loss: 0.7019 | Val Loss: 0.6648  *
    Epoch  1 | Train Loss: 0.6543 | Val Loss: 0.6411  *
    Epoch  2 | Train Loss: 0.5814 | Val Loss: 0.6248  *
    Epoch  3 | Train Loss: 0.5086 |

100%|██████████| 20.5M/20.5M [00:00<00:00, 83.2MB/s]


    Epoch  0 | Train Loss: 0.6267 | Val Loss: 0.5048  *
    Epoch  1 | Train Loss: 0.4714 | Val Loss: 0.3594  *
    Epoch  2 | Train Loss: 0.3295 | Val Loss: 0.2545  *
    Epoch  3 | Train Loss: 0.2505 | Val Loss: 0.1726  *
    Epoch  4 | Train Loss: 0.1952 | Val Loss: 0.1340  *
    Epoch  5 | Train Loss: 0.1685 | Val Loss: 0.1239  *
    Epoch  6 | Train Loss: 0.1457 | Val Loss: 0.1217  *
    Epoch  7 | Train Loss: 0.0841 | Val Loss: 0.1276
    Epoch  8 | Train Loss: 0.0839 | Val Loss: 0.1445
    Epoch  9 | Train Loss: 0.1106 | Val Loss: 0.1289
    Epoch 10 | Train Loss: 0.1050 | Val Loss: 0.1375
    Epoch 11 | Train Loss: 0.1049 | Val Loss: 0.1328
    Early stopping triggered at epoch 11 (Best epoch: 6, Val Loss: 0.1217)
Saved checkpoint: efficientnet_b0__K2N__size190__seed42.pth (95.4s)

--- Running [efficientnet_b0__N2K__size190__seed42__fold0] ---
Train samples: 127 | Val samples: 33 | Batch size: 16
    Epoch  0 | Train Loss: 0.7186 | Val Loss: 0.7351  *
    Epoch  1 | Train Loss:

## 5. Full Experimental Scale-Out

In [6]:
RUN_FULL = True

if RUN_FULL:
    print("=========================================")
    print("Launching Full Design Training Matrix")
    print("=========================================")

    full_records = []
    skipped_count = 0
    trained_count = 0

    def _run_and_track(arch, direction, size, seed, fold=None):
        global skipped_count, trained_count
        tag = (f"{arch}__K2N__size{size}__seed{seed}" if direction == "K2N"
               else f"{arch}__N2K__size190__seed{seed}__fold{fold if fold is not None else 0}")
        already_exists = (config.CHECKPOINTS_DIR / f"{tag}.pth").exists()
        rec = run_one(arch, direction, size=size, seed=seed, fold=fold, verbose=False)
        if already_exists:
            skipped_count += 1
        else:
            trained_count += 1
        return rec

    # Direction 1: K2N (2 architectures x 3 sizes x N seeds)
    for seed in config.SEEDS:
        for arch in config.ARCHITECTURES:
            for size in config.MATCHED_SIZES:
                full_records.append(_run_and_track(arch, "K2N", size=size, seed=seed))

    # Direction 2: N2K (2 architectures x 5 folds x N seeds)
    for seed in config.SEEDS:
        for arch in config.ARCHITECTURES:
            for fold in range(config.N_FOLDS):
                full_records.append(_run_and_track(arch, "N2K", size=190, seed=seed, fold=fold))

    fdf = pd.DataFrame(full_records)
    log_path = config.RESULTS_DIR / "03_training_log.csv"
    fdf.to_csv(log_path, index=False)

    print(f"\nCompleted {len(full_records)} configs total: {trained_count} newly trained, {skipped_count} skipped (already existed).")
    print(f"Saved log to {log_path.name}")
    print("\nFit counts by architecture and direction:")
    print(fdf.groupby(["arch", "direction"]).size().to_string())
else:
    print("\nRUN_FULL is set to False. Toggle RUN_FULL = True to execute the full study matrix.")


Launching Full Design Training Matrix
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 47.2MB/s]


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 139MB/s] 



Completed 96 configs total: 48 newly trained, 48 skipped (already existed).
Saved log to 03_training_log.csv

Fit counts by architecture and direction:
arch             direction
efficientnet_b0  K2N          18
                 N2K          30
mobilenet_v2     K2N          18
                 N2K          30


**Next:** notebook 04 runs each checkpoint on its cross-population target.